<a href="https://colab.research.google.com/github/Hiregayatri/ASSIGNMENT-NO-6/blob/main/flight_price.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **import Libraries**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import joblib

In [ ]:
df=pd.read_excel("/content/drive/MyDrive/Colab Notebooks/dataset/Flight_Ticket.xlsx")

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Airline          10683 non-null  object
 1   Date_of_Journey  10683 non-null  object
 2   Source           10683 non-null  object
 3   Destination      10683 non-null  object
 4   Route            10682 non-null  object
 5   Dep_Time         10683 non-null  object
 6   Arrival_Time     10683 non-null  object
 7   Duration         10683 non-null  object
 8   Total_Stops      10682 non-null  object
 9   Additional_Info  10683 non-null  object
 10  Price            10683 non-null  int64 
dtypes: int64(1), object(10)
memory usage: 918.2+ KB


In [ ]:
df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302


In [ ]:
df.describe()

,Price
count,10683.000000
mean,9087.064121
std,4611.359167
min,1759.000000
25%,5277.000000
50%,8372.000000
75%,12373.000000
max,79512.000000


In [ ]:
df.isnull().sum()

,0
Airline,0
Date_of_Journey,0
Source,0
Destination,0
Route,1
Dep_Time,0
Arrival_Time,0
Duration,0
Total_Stops,1
Additional_Info,0


In [ ]:
df['Journey_Day']=pd.to_datetime(df['Date_of_Journey'],format='%d/%m/%Y').dt.day
df['Journey_Month']=pd.to_datetime(df['Date_of_Journey'],format='%d/%m/%Y').dt.month
df.drop('Date_of_Journey',axis=1,inplace=True)

In [ ]:
df['Dep_hour']=pd.to_datetime(df['Dep_Time']).dt.hour
df['Dep_minute']=pd.to_datetime(df['Dep_Time']).dt.minute

/tmp/ipython-input-818/935367083.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Dep_hour']=pd.to_datetime(df['Dep_Time']).dt.hour
/tmp/ipython-input-818/935367083.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Dep_minute']=pd.to_datetime(df['Dep_Time']).dt.minute


In [ ]:
df.drop('Dep_Time',axis=1,inplace=True)

In [ ]:
df['Arrival_hour']=pd.to_datetime(df['Arrival_Time']).dt.hour
df['Arrival_Minute']=pd.to_datetime(df['Arrival_Time']).dt.minute

/tmp/ipython-input-818/1157405588.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Arrival_hour']=pd.to_datetime(df['Arrival_Time']).dt.hour
/tmp/ipython-input-818/1157405588.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Arrival_Minute']=pd.to_datetime(df['Arrival_Time']).dt.minute


In [ ]:
df.drop('Arrival_Time',axis=1,inplace=True)

In [ ]:
df['duration_hours']=df['Duration'].str.extract(r'(\d+)h')
df['duration_hours']=df['duration_hours'].fillna(0).astype('int')

In [ ]:
df['duration_minutes']=df['Duration'].str.extract(r'(\d+)m')
df['duration_minutes']=df['duration_minutes'].fillna(0).astype('int')

In [ ]:
df.drop('Duration',axis=1,inplace=True)

In [ ]:
df.Total_Stops.value_counts()

,count
Total_Stops,
1 stop,5625
non-stop,3491
2 stops,1520
3 stops,45
4 stops,1


In [ ]:
df['Total_Stops']=df['Total_Stops'].replace({
    'non-stop' :0,
    '1 stop': 1,
    '2 stops':2,
    '3 stops':3,
    '4 stops':4
})

/tmp/ipython-input-818/4031991734.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Total_Stops']=df['Total_Stops'].replace({


In [ ]:
df.drop('Route', axis=1, inplace=True)

x = df.drop('Price', axis=1)
y = df['Price']

In [ ]:
num_col=x.select_dtypes(include=['int32','int64','float64'])
cat_col=x.select_dtypes(include=['object'])

In [ ]:
numeric_transformer=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median'))])

In [ ]:
categorical_transformer=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(handle_unknown='ignore'))
])

In [ ]:
preprocessor=ColumnTransformer(
    transformers=[
        ('num',numeric_transformer,num_col.columns.tolist()),
        ('cat',categorical_transformer,cat_col.columns.tolist())
    ]
)

In [ ]:
pipeline=Pipeline(steps=[
                  ('Preprocessing',preprocessor),
                  ('model',RandomForestRegressor(n_estimators=200,random_state=42))
                  ])

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
pipeline.fit(x_train,y_train)

Pipeline(steps=[('Preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Total_Stops', 'Journey_Day',
                                                   'Journey_Month', 'Dep_hour',
                                                   'Dep_minute', 'Arrival_hour',
                                                   'Arrival_Minute',
                                                   'duration_hours',
                                                   'duration_minutes']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Airline', 'Source',
                                                   'Destination',
                                                   'Additional_Info'])])),
                ('model',
                 RandomForestRegressor(n_estimators=200, random_state=42))])

In [ ]:
y_pred=pipeline.predict(x_test)

In [ ]:
print('r2score:',r2_score(y_test,y_pred))
print('Mean Absolute Error:',mean_absolute_error(y_test,y_pred))

r2score: 0.871152535236293
Mean Absolute Error: 654.8078860758127


In [ ]:
joblib.dump(pipeline,'flight_price_pipeline.pkl')

['flight_price_pipeline.pkl']

Install Streamlit

In [ ]:
print(sklearn.__version__)

NameError: name 'sklearn' is not defined

In [ ]:
pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 94.4 MB/s eta 0:00:00


create app.py file

In [ ]:
from google.colab import files
files.download("flight_price_pipeline.pkl")